In [ ]:
import sys
import community as louvain_community
import networkx as nx
import itertools
import infomap
from collections import Counter

sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/pipeline/cd_cluster.py'
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'

%run 'common.py'

import altair.vega.v5 as alt
import cdlib
import numpy as np
from cdlib.readwrite import read_community_json

In [ ]:
def sortable_value(text):
    match = re.fullmatch('\d+([a-z]*)', text, flags=re.IGNORECASE)
    if not match:
        return text.zfill(4)
    extra_len = len(match[1])
    return text.zfill(4 + extra_len)

# Steuerrecht

In [ ]:
clustering = get_clustering_result(
    "2019-01-01_0-0_1-0_-1_o-2-0_t-paragraph_a-louvain_m1-0_s0_c1000.json", 
    'de', 
    'seqitems',
    path_prefix='../'
)

In [ ]:
H = hierarchy_graph(clustering.graph)

In [ ]:
steuerrechts_nodes = [c for c in clustering.communities if any(True for n in c if '_EStG_' in n)][0]
seqitem_nodes = [
    n
    for l in steuerrechts_nodes 
    for n in nx.predecessor(H, l)
#     if H.nodes[n].get('type') =='seqitem'
]

In [ ]:
proprecessed_graph = nx.read_gpickle('../../legal-networks-data/de/10_preprocessed_graph/2019-01-01_1-0_1-0_0.gpickle.gz')

In [ ]:
proprecessed_graph = nx.read_gpickle(
    '../../legal-networks-data/de/10_preprocessed_graph/2019-01-01_1-0_1-0_0_o-2-0_t-paragraph.gpickle.gz')

In [ ]:
for u, v, k, t in proprecessed_graph.edges(keys=True, data='edge_type'):
    if t == 'sequence':
        proprecessed_graph.edges[u, v, k]['weight'] *= 6


In [ ]:
steuerrecht_G = proprecessed_graph.subgraph(seqitem_nodes)
steuerrecht_Gw = make_weighted(steuerrecht_G)
H = compile_source_graph(steuerrecht_Gw, 'louvain')
steuerrecht_clustering, D = cdlib_custom_algorithms.louvain(
    H,
    weight="weight",
    seed=0,
    resolution=2,
    return_tree=True,
)

In [ ]:
def format_keys(key_list, gesetz):
    full_order = gesetz_nr_dict[gesetz]
    key_list = sorted(cite_keys_formatted[gesetz], key=lambda x: sortable_value(x))
    in_range = [
        full_order[full_order.index(b)-1] == a and full_order[full_order.index(b)+1] == c
        for a, b, c in zip(key_list[:-2], key_list[1:-1], key_list[2:])
    ]
    in_range = [False, *in_range, False]
    key_list = [
        '-' if sub else k 
        for k, sub in zip(key_list, in_range)
    ]
    key_list = [k for idx, k in enumerate(key_list) if idx == 0 or k != '-' or key_list[idx-1] != '-']
    key_list = ' '.join(key_list).replace(' - ', '-').replace(' ', ', ')
    return key_list

In [ ]:
gesetz_nr_dict = defaultdict(list)
for v in nx.get_node_attributes(steuerrecht_clustering.graph, 'citekey').values():
    gesetz, nr = v.split('_')
    gesetz_nr_dict[gesetz].append(nr)

gesetz_nr_dict = {gesetz: sorted(nrs, key=lambda n: sortable_value(n)) for gesetz, nrs in gesetz_nr_dict.items()}

In [ ]:
soreted_communities = sorted(steuerrecht_clustering.communities, key=lambda nodes: -sum(H.nodes[node]['tokens_n'] for node in nodes))

In [ ]:
df_data = []

single_law_cluster = []

for idx, com in enumerate(soreted_communities):
    
    cite_keys = [
        (
            steuerrecht_clustering.graph.nodes[n]['citekey'].split('_'), 
            steuerrecht_clustering.graph.nodes[n].get('law_name') or ''
        )
        for n in com
        if 'citekey' in steuerrecht_clustering.graph.nodes[n]
    ]
    cite_keys_formatted = defaultdict(list)
    lawnames = {}
    for (gesetz, nummer), lawname in sorted(cite_keys):
        lawnames[gesetz] = lawname
        cite_keys_formatted[gesetz].append(nummer)
    
#     if len(cite_keys_formatted) == 1 and len(cite_keys_formatted[gesetz]) == abk_counter[gesetz]:
#         single_law_cluster.append([str(idx+1), gesetz])
#     else:
    print('\n\nCLUSTER', idx)
    for gesetz in sorted(cite_keys_formatted.keys(), key=lambda x: -len(cite_keys_formatted[x])):

#             if len(cite_keys_formatted[gesetz]) == abk_counter[gesetz]:
#                 contents = [f'\n\t===== Alle {abk_counter[gesetz]} Elemente =====']
#             else:
        contents = format_keys(cite_keys_formatted[gesetz], gesetz)
        print('-', gesetz, ':', contents, '\n\t', lawnames[gesetz])
        df_data.append([str(idx+1), gesetz,  contents])

In [ ]:
tex = (
    '\\begin{tabular}{p{1.2cm}p{8.5cm}}\n'
    '\\toprule\n'
    'Gruppe & Gesetz (Paragraphen/Artikel) \\\\\n'
    '\\midrule\n'
)

last_group = None
group_counter = 0
for group, gesetz, nrs in df_data:
    if last_group != group:
        group_counter += 1
        if group_counter > 6:
            break
        if last_group is not None:
            tex += ' \\\\ \n'
        last_group = group
        tex += group + ' & '
        
    tex += '\\textbf{' + gesetz + '} (' + nrs + ') '
    
tex += '\\\\ \n\\bottomrule\n\\end{tabular}\n'

# tex += 'Folgende Gesetze sind vollständig einer Gruppe zugeordnet, die wiederum genau ein Gesetz enthält:\n\n'

# for nr, gesetz in single_law_cluster:
#     tex += f'{gesetz}, '

# tex= tex[:-2]

with open('../tables/meso_communities_2019_steuerrecht_de.tex', 'w') as f:
    f.write(tex)

In [ ]:
print(tex)